<a href="https://colab.research.google.com/github/NnamdiOdozi/NEBIUS_MAR_2026/blob/main/AI_Agents_Week2_Tools_V2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tools for AI Agents - Code Cells

Extracted from the Nebius Academy Module 1 Week 2 slides.

## 1. CLI Tools

### Cell 1: Environment Setup
*(Slide 21)*

In [1]:
import os, subprocess, json
from pathlib import Path
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=userdata.get('NEBIUS_KEY').strip()
)
WORKSPACE = Path("/tmp/edinburgh_workspace")
WORKSPACE.mkdir(exist_ok=True)
(WORKSPACE / "venues.txt").write_text(
    "The Albanach,capacity=180,vegan=yes,address=2 Hunter Square Edinburgh\n"
    "The Bow Bar,capacity=80,vegan=yes,address=80 West Bow Edinburgh\n"
    "The Guilford Arms,capacity=200,vegan=no,address=1 West Register Street Edinburgh\n"
    "The Hanging Bat,capacity=70,vegan=yes,address=133 Lothian Road Edinburgh\n"
    "The Grain Store,capacity=170,vegan=no,address=30 Victoria Street Edinburgh\n"
    "The Haymarket Vaults,capacity=160,vegan=yes,address=1 Dalry Road Edinburgh\n"
)
print("Workspace ready:", WORKSPACE)
print("Venue file created:", (WORKSPACE / "venues.txt").read_text())

Workspace ready: /tmp/edinburgh_workspace
Venue file created: The Albanach,capacity=180,vegan=yes,address=2 Hunter Square Edinburgh
The Bow Bar,capacity=80,vegan=yes,address=80 West Bow Edinburgh
The Guilford Arms,capacity=200,vegan=no,address=1 West Register Street Edinburgh
The Hanging Bat,capacity=70,vegan=yes,address=133 Lothian Road Edinburgh
The Grain Store,capacity=170,vegan=no,address=30 Victoria Street Edinburgh
The Haymarket Vaults,capacity=160,vegan=yes,address=1 Dalry Road Edinburgh



### Cell 2: The CLI Tool
*(Slides 22–23)*

In [2]:
MAX_OUTPUT = 8_192

# 8 KB cap on stdout returned to the model

def search_venue_notes(query: str) -> dict:
    """Search local venue notes for a keyword using grep."""
    venue_file = WORKSPACE / "venues.txt"
    if not venue_file.exists():
        return {"success": False, "error": "venues.txt not found in workspace"}
    try:
        result = subprocess.run(
            ["grep", "-i", "--", query, str(venue_file)],
            capture_output=True, text=True, timeout=5
        )
        lines = [l.strip() for l in result.stdout[:MAX_OUTPUT].splitlines() if l.strip()]
        return {"success": True, "matches": lines,
                "count": len(lines), "query": query}
    except subprocess.TimeoutExpired:
        return {"success": False, "error": "Search timed out after 5 seconds"}
    except Exception as e:
        return {"success": False, "error": str(e)}

TOOLS = [{"type": "function", "function": {
    "name": "search_venue_notes",
    "description": "Search local Edinburgh venue notes for a keyword. "
                   "Use this to find pubs by name, capacity, or dietary options. "
                   "Do NOT use this for live data — it reads a local file only.",
    "parameters": {"type": "object",
                   "properties": {"query": {"type": "string",
                                            "description": "Search term, e.g. 'vegan', '160', 'Haymarket'"}},
                   "required": ["query"]}}}]
print("CLI tool defined.")

CLI tool defined.


### Cell 3: The Agent Loop
*(Slide 24)*

In [7]:
def run_cli_agent(task: str) -> str:
    messages = [{"role": "user", "content": task}]
    for _ in range(5):
        resp = client.chat.completions.create(
            model="meta-llama/Llama-3.3-70B-Instruct",
            messages=messages, tools=TOOLS, max_tokens=300
        )
        msg = resp.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            result = search_venue_notes(**args)
            print(f" grep('{args['query']}') → {result['count']} match(es)")
            messages.append({
                "role": "tool", "tool_call_id": tc.id,
                "content": json.dumps(result)
            })
    return "Max turns reached.", messages

print(run_cli_agent(
    "Search our venue notes for Edinburgh pubs that have vegan options "
    "and can fit 160 people. List the names and addresses."
))

 grep('vegan 160') → 0 match(es)
This function call will search your venue notes for Edinburgh pubs that have vegan options and a capacity of 160 people, then it will list the names and addresses of matching venues.


## 2. In-Process Functions

### Cell 1: Tool Implementations
*(Slides 30–31)*

In [ ]:
import json
from datetime import datetime
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=userdata.get('NEBIUS_KEY').strip()
)
VENUES = {
    "The Albanach": {"capacity": 180, "vegan": True, "status": "available"},
    "The Guilford Arms": {"capacity": 200, "vegan": False, "status": "available"},
    "The Haymarket Vaults": {"capacity": 160, "vegan": True, "status": "available"},
    "The Bow Bar": {"capacity": 80, "vegan": True, "status": "full"},
}

def check_pub_availability(pub_name: str, required_capacity: int, requires_vegan: bool) -> dict:
    venue = VENUES.get(pub_name)
    if not venue:
        return {"success": False, "error": f"Venue not found: {pub_name}", "error_code": "NOT_FOUND"}
    return {
        "success": True, "pub_name": pub_name,
        "capacity": venue["capacity"], "vegan": venue["vegan"],
        "status": venue["status"],
        "meets_all_constraints": (venue["capacity"] >= required_capacity
                                  and (not requires_vegan or venue["vegan"])
                                  and venue["status"] == "available")
    }

### Cell 2: More Tool Implementations
*(Slides 32–33)*

In [ ]:
def calculate_catering_cost(guests: int, price_per_head: float) -> dict:
    if guests <= 0 or price_per_head < 0:
        return {"success": False, "error": "guests and price_per_head must be positive"}
    return {
        "success": True, "guests": guests,
        "price_per_head_gbp": price_per_head,
        "total_cost_gbp": round(guests * price_per_head, 2)
    }

def get_booking_deadline(event_date: str) -> dict:
    try:
        date = datetime.strptime(event_date, "%Y-%m-%d")
        deadline = date.replace(hour=17, minute=0, second=0)
        now = datetime.now()
        hours = (deadline - now).total_seconds() / 3600
        return {"success": True, "deadline": deadline.strftime("%H:%M"),
                "hours_remaining": round(hours, 1),
                "urgent": 0 < hours < 2, "expired": hours <= 0}
    except ValueError:
        return {"success": False, "error": "event_date must be YYYY-MM-DD format"}

TOOL_MAP = {
    "check_pub_availability": check_pub_availability,
    "calculate_catering_cost": calculate_catering_cost,
    "get_booking_deadline": get_booking_deadline,
}
print("Tools ready:", list(TOOL_MAP.keys()))

Tools ready: ['check_pub_availability', 'calculate_catering_cost', 'get_booking_deadline']


### Cell 3: JSON Schemas
*(Slides 34–35)*

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "check_pub_availability",
        "description": ("Check if a named Edinburgh pub meets capacity and dietary "
                        "requirements. Returns status, capacity, and vegan availability. "
                        "Use AFTER finding candidates with search. "
                        "Do NOT use for weather or cost — use the dedicated tools."),
        "parameters": {"type": "object", "properties": {
            "pub_name": {"type": "string", "description": "Exact pub name from our shortlist"},
            "required_capacity": {"type": "integer", "description": "Minimum guest count the venue must hold"},
            "requires_vegan": {"type": "boolean", "description": "True if vegan menu is mandatory"}
        }, "required": ["pub_name", "required_capacity", "requires_vegan"]}}},
    {"type": "function", "function": {
        "name": "calculate_catering_cost",
        "description": "Estimate total catering cost for the event. Use after confirming a venue.",
        "parameters": {"type": "object", "properties": {
            "guests": {"type": "integer"},
            "price_per_head": {"type": "number", "description": "Cost per guest in GBP"}
        }, "required": ["guests", "price_per_head"]}}}
]

TOOLS.extend([
    {"type": "function", "function": {
        "name": "get_booking_deadline",
        "description": "Calculate hours remaining until the 5 PM booking cutoff today.",
        "parameters": {"type": "object", "properties": {
            "event_date": {"type": "string", "description": "Event date in YYYY-MM-DD format"}
        }, "required": ["event_date"]}}}
])
print(f"{len(TOOLS)} tool schemas registered.")

3 tool schemas registered.


### Cell 4: The Explicit Agent Loop
*(Slide 36)*

In [ ]:
def run_agent(task: str, max_turns: int = 8) -> str:
    messages = [{"role": "user", "content": task}]
    for turn in range(max_turns):
        resp = client.chat.completions.create(
            model="Qwen/Qwen3-235B-A22B-Instruct-2507",
            messages=messages, tools=TOOLS, max_tokens=400
        )
        msg = resp.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            fn = TOOL_MAP.get(tc.function.name)
            result = fn(**args) if fn else {"success": False, "error": f"Unknown tool: {tc.function.name}"}
            print(f" [{tc.function.name}] → {result}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    return "Max turns reached."

print(run_agent(
    "For tonight's event (2026-03-28), check The Albanach and The Haymarket Vaults "
    "for 160 guests with vegan options. If one works, estimate catering at £35/head. "
    "Also tell me if we're still within the 5 PM booking deadline."
))

 [check_pub_availability] → {'success': True, 'pub_name': 'The Albanach', 'capacity': 180, 'vegan': True, 'status': 'available', 'meets_all_constraints': True}
 [check_pub_availability] → {'success': True, 'pub_name': 'The Haymarket Vaults', 'capacity': 160, 'vegan': True, 'status': 'available', 'meets_all_constraints': True}
 [get_booking_deadline] → {'success': True, 'deadline': '17:00', 'hours_remaining': -6.2, 'urgent': False, 'expired': True}
 [calculate_catering_cost] → {'success': True, 'guests': 160, 'price_per_head_gbp': 35, 'total_cost_gbp': 5600}
Both **The Albanach** and **The Haymarket Vaults** are available for tonight's event and meet your requirements:

- Capacity for 160 guests: ✅  
- Vegan menu available: ✅  
  - The Albanach holds up to 180 guests.
  - The Haymarket Vaults fits exactly 160 guests.

However, the **5 PM booking deadline has already passed** (it was 5:00 PM today, and the current time is now later), so bookings for tonight can no longer be made.

**Cate

## 3. External APIs

### Cell 1: HTTP Helper + Geocoder
*(Slides 43–44)*

In [ ]:
import time, urllib.request, urllib.parse, json
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    base_url="https://api.tokenfactory.nebius.com/v1/",
    api_key=userdata.get('NEBIUS_KEY').strip()
)

def http_get(url: str, params: dict = None, timeout: float = 8.0, max_retries: int = 3) -> dict:
    """GET with exponential backoff. Returns parsed JSON."""
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    for attempt in range(max_retries + 1):
        try:
            req = urllib.request.Request(url, headers={"Accept": "application/json"})
            with urllib.request.urlopen(req, timeout=timeout) as resp:
                return json.loads(resp.read().decode("utf-8"))
        except Exception as e:
            if attempt == max_retries:
                raise
            time.sleep(0.5 * (2 ** attempt))

def geocode_location(location: str) -> dict:
    """Convert a place name to lat/lon using Open-Meteo geocoding API."""
    try:
        data = http_get("https://geocoding-api.open-meteo.com/v1/search",
                        {"name": location, "count": 1, "language": "en"})
        if not data.get("results"):
            return {"success": False, "error": f"Location not found: '{location}'", "error_code": "NOT_FOUND"}
        r = data["results"][0]
        return {"success": True, "name": r["name"],
                "country": r.get("country"), "latitude": r["latitude"],
                "longitude": r["longitude"], "timezone": r.get("timezone")}
    except Exception as e:
        return {"success": False, "error": f"Geocoding error: {e}", "error_code": "NETWORK_ERROR"}

### Cell 2: Weather Tool + Schemas
*(Slides 45–46)*

In [ ]:
WMO = {0: "Clear sky", 1: "Mainly clear", 2: "Partly cloudy", 3: "Overcast",
       45: "Fog", 61: "Light rain", 63: "Moderate rain", 65: "Heavy rain",
       80: "Rain showers", 95: "Thunderstorm"}

def get_current_weather(latitude: float, longitude: float) -> dict:
    """Fetch current weather from Open-Meteo. No API key required."""
    try:
        data = http_get("https://api.open-meteo.com/v1/forecast", {
            "latitude": latitude, "longitude": longitude, "forecast_days": 1,
            "current": "temperature_2m,wind_speed_10m,precipitation,weather_code"
        })
        curr = data.get("current", {})
        code = curr.get("weather_code", -1)
        return {"success": True, "temperature_c": curr.get("temperature_2m"),
                "wind_kph": curr.get("wind_speed_10m"),
                "precipitation_mm": curr.get("precipitation"),
                "description": WMO.get(code, f"Code {code}"),
                "outdoor_suitable": code in {0, 1, 2}}
    except Exception as e:
        return {"success": False, "error": f"Weather API error: {e}", "error_code": "UPSTREAM_ERROR"}

TOOLS = [
    {"type": "function", "function": {
        "name": "geocode_location",
        "description": ("Convert a place name to geographic coordinates. "
                        "Call this FIRST if you only have a location name, not coordinates. "
                        "Do NOT use for weather — call get_current_weather after geocoding."),
        "parameters": {"type": "object",
                       "properties": {"location": {"type": "string",
                                                   "description": "Place name, e.g. 'Edinburgh Old Town'"}},
                       "required": ["location"]}}},
    {"type": "function", "function": {
        "name": "get_current_weather",
        "description": ("Get live weather for a lat/lon coordinate. "
                        "Call geocode_location first if you only have a place name."),
        "parameters": {"type": "object",
                       "properties": {
                           "latitude": {"type": "number"},
                           "longitude": {"type": "number"}},
                       "required": ["latitude", "longitude"]}}}
]
print("API tools ready.")

API tools ready.


### Cell 3: Running the API Agent
*(Slides 47–48)*

In [ ]:
TOOL_MAP = {"geocode_location": geocode_location,
            "get_current_weather": get_current_weather}

def run_api_agent(task: str) -> str:
    messages = [{"role": "user", "content": task}]
    for _ in range(6):
        resp = client.chat.completions.create(
            model="meta-llama/Llama-3.3-70B-Instruct",
            messages=messages, tools=TOOLS, max_tokens=300
        )
        msg = resp.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:
            return msg.content
        for tc in msg.tool_calls:
            args = json.loads(tc.function.arguments)
            fn = TOOL_MAP.get(tc.function.name)
            result = fn(**args) if fn else {"success": False, "error": "Unknown tool"}
            print(f" [{tc.function.name}({args})]")
            print(f" → {json.dumps(result)[:120]}")
            messages.append({"role": "tool", "tool_call_id": tc.id, "content": json.dumps(result)})
    return "Max turns reached."

print(run_api_agent(
    "We are hosting an event in Edinburgh Old Town tonight. "
    "Check the current weather and tell us whether the outdoor terrace "
    "at our venue is a good idea, or if we should book the indoor space."
))

 [get_current_weather({'latitude': 55.9492239, 'longitude': -3.1924731})]
 → {"success": true, "temperature_c": 4.4, "wind_kph": 13.0, "precipitation_mm": 0.0, "description": "Clear sky", "outdoor_
Based on the current weather, it seems suitable to use the outdoor terrace at your venue. The temperature is around 4.4°C, with a gentle wind of 13.0 kph and no precipitation. The description mentions a clear sky, which suggests a pleasant evening. Therefore, you may not need to book the indoor space.


## 4. MCP Servers

### Cell 0: Install MCP Dependencies

In [ ]:
!pip install -q "mcp[cli]" nest_asyncio

### Cell 1: Write MCP Server to Disk
*(Slides 55–57)*

This cell writes `mcp_venue_server.py` to disk using `%%writefile`.
The MCP client (Cell 2) will spawn it automatically via stdio transport — no manual launch needed.

In [ ]:
%%writefile mcp_venue_server.py
from mcp.server.fastmcp import FastMCP
from datetime import datetime
import json

mcp = FastMCP("EdinburghVenueServer")

VENUES = {
    "The Albanach": {"capacity": 180, "vegan": True, "status": "available", "address": "2 Hunter Square, Edinburgh"},
    "The Guilford Arms": {"capacity": 200, "vegan": False, "status": "available", "address": "1 West Register Street, Edinburgh"},
    "The Haymarket Vaults": {"capacity": 160, "vegan": True, "status": "available", "address": "1 Dalry Road, Edinburgh"},
    "The Bow Bar": {"capacity": 80, "vegan": True, "status": "full", "address": "80 West Bow, Edinburgh"},
}

@mcp.tool()
def search_venues(min_capacity: int, requires_vegan: bool) -> str:
    """Search Edinburgh venues by minimum capacity and dietary requirements."""
    results = [
        {"name": name, **info}
        for name, info in VENUES.items()
        if info["capacity"] >= min_capacity
        and (not requires_vegan or info["vegan"])
        and info["status"] == "available"
    ]
    return json.dumps({"matches": results, "count": len(results)})

@mcp.tool()
def get_venue_details(pub_name: str) -> str:
    """Get full details for a specific Edinburgh venue by name."""
    venue = VENUES.get(pub_name)
    if not venue:
        return json.dumps({"success": False, "error": f"Venue not found: '{pub_name}'"})
    return json.dumps({"success": True, "name": pub_name, **venue})

@mcp.tool()
def check_booking_window() -> str:
    """Return hours remaining until the 5 PM booking deadline today."""
    now = datetime.now()
    deadline = now.replace(hour=17, minute=0, second=0, microsecond=0)
    hours = (deadline - now).total_seconds() / 3600
    return json.dumps({
        "hours_remaining": round(max(hours, 0), 1),
        "deadline": "17:00",
        "urgent": 0 < hours < 2,
        "expired": hours <= 0
    })

if __name__ == "__main__":
    mcp.run()

Writing mcp_venue_server.py


### Cell 2: MCP Client Agent
*(Slides 58–59)*

The client spawns `mcp_venue_server.py` as a subprocess via stdio transport, so the server starts and stops automatically.

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import asyncio, json, sys
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from openai import OpenAI
from google.colab import userdata

llm = OpenAI(base_url="https://api.tokenfactory.nebius.com/v1/",
             api_key=userdata.get('NEBIUS_KEY').strip())

async def mcp_agent(query: str) -> str:
    params = StdioServerParameters(command=sys.executable, args=["mcp_venue_server.py"])
    # errlog=asyncio.subprocess.PIPE avoids Colab's fileno error
    # (Colab's virtual stderr has no real file descriptor)
    async with stdio_client(params, errlog=asyncio.subprocess.PIPE) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            # Dynamic discovery — no manual schema registration
            tool_list = await session.list_tools()
            tools = [{"type": "function", "function": {"name": t.name, "description": t.description or "", "parameters": t.inputSchema}} for t in tool_list.tools]
            print(f"Discovered {len(tools)} tools: {[t['function']['name'] for t in tools]}")

            messages = [{"role": "user", "content": query}]
            for _ in range(8):
                resp = llm.chat.completions.create(
                    model="Qwen/Qwen3-235B-A22B-Instruct-2507",
                    messages=messages, tools=tools, max_tokens=300)
                msg = resp.choices[0].message
                messages.append(msg)
                if not msg.tool_calls:
                    return msg.content
                for tc in msg.tool_calls:
                    args = json.loads(tc.function.arguments)
                    result = await session.call_tool(tc.function.name, args)
                    output = result.content[0].text if result.content else "{}"
                    print(f" MCP:{tc.function.name} → {output[:100]}")
                    messages.append({"role":"tool","tool_call_id":tc.id,"content":output})
            return "Loop ended."

# Run the agent
print(asyncio.run(mcp_agent("Which Edinburgh venues fit 160 vegan guests, and how long until the booking deadline?")))


Discovered 3 tools: ['search_venues', 'get_venue_details', 'check_booking_window']
 MCP:search_venues → {"matches": [{"name": "The Albanach", "capacity": 180, "vegan": true, "status": "available", "addres
 MCP:check_booking_window → {"hours_remaining": 0, "deadline": "17:00", "urgent": false, "expired": true}
The following Edinburgh venues can accommodate 160 vegan guests:

1. **The Albanach**  
   - Capacity: 180  
   - Vegan options: Yes  
   - Status: Available  
   - Address: 2 Hunter Square, Edinburgh  

2. **The Haymarket Vaults**  
   - Capacity: 160  
   - Vegan options: Yes  
   - Status: Available  
   - Address: 1 Dalry Road, Edinburgh  

Regarding the booking deadline:  
The 5 PM booking window has already expired today (0 hours remaining), so new bookings cannot be made until the next available booking period.


## 5. A2A Agent Delegation
### Cell 0: Install A2A Dependencies

In [ ]:
!pip install -q flask requests

### Cell 1: A2A Server — Define, Launch, and Manage
*(Slides 66–68)*

The next two cells are **safe to re-run**. On each run it:
1. Shuts down any previously running server thread via `_httpd.shutdown()`
2. Scans for a free port starting at 8080
3. Launches the Flask server on a daemon thread using `make_server()`

`make_server()` is Werkzeug's factory function — the same one `app.run()` calls
internally. The difference is that it **returns the server object** so we can
call `.shutdown()` on it later. `.shutdown()` is inherited from Python's
`socketserver.BaseServer` — it sets a flag that makes `serve_forever()` exit.

In [ ]:
import socket

# ── 1. Shut down previous server if it exists ───────────────────────────
if '_httpd' in dir() and _httpd is not None and 'server_thread' in globals():
    _httpd.shutdown()
    server_thread.join(timeout=3)
    print(f"Stopped previous server on port {SERVER_PORT}.")
    _httpd = None

# ── 2. Find a free port (start at 8080, increment if occupied) ──────────
SERVER_PORT = SERVER_PORT if 'SERVER_PORT' in globals() else 8000
while True:
    try:
        with socket.create_connection(("localhost", SERVER_PORT), timeout=0.3):
            print(f"Port {SERVER_PORT} is occupied, trying {SERVER_PORT + 1}...")
            SERVER_PORT += 1
    except OSError:
        break
print(f"Using port {SERVER_PORT}")

Using port 8000


In [ ]:
import threading, time, socket, uuid, json as _json
from flask import Flask, jsonify, request
from werkzeug.serving import make_server


# ── 3. Define the Flask A2A server ──────────────────────────────────────
app = Flask(__name__)
TASKS = {}
AGENT_CARD = {
    "name": "EdinburghVenueBookingAgent",
    "description": "Handles venue booking negotiations for Edinburgh events.",
    "version": "1.0.0",
    "url": f"http://localhost:{SERVER_PORT}/a2a",
    "capabilities": {"streaming": False, "pushNotifications": False},
    "skills": [{
        "id": "book_venue",
        "name": "Book Venue",
        "description": ("Confirm a venue booking for a given guest count and date. "
                        "Returns a confirmation with deposit terms."),
        "inputModes": ["text"], "outputModes": ["text", "data"]
    }]
}

@app.get("/.well-known/agent.json")
def agent_card():
    """Serve the Agent Card for discovery."""
    return jsonify(AGENT_CARD)

def process_booking(task_id: str, text: str):
    """Background processing: validate request and confirm booking."""
    time.sleep(1)
    confirmed = "160" in text and "vegan" in text.lower()
    result = (
        "CONFIRMED: The Haymarket Vaults \u2014 160 guests, vegan menu included. "
        "Deposit: \u00a3200 required by 5 PM today. Ref: HV-2603."
        if confirmed else
        "FAILED: No venue available matching all constraints provided."
    )
    TASKS[task_id].update({
        "status": {"state": "completed"},
        "artifacts": [{"type": "text", "name": "booking_confirmation",
                       "parts": [{"type": "text", "text": result}]}]
    })

@app.post("/a2a/tasks/send")
def tasks_send():
    body = request.get_json(force=True)
    task_id = body.get("id") or str(uuid.uuid4())
    parts = body.get("message", {}).get("parts", [])
    text = " ".join(p.get("text", "") for p in parts if "text" in p)
    TASKS[task_id] = {"id": task_id, "status": {"state": "submitted"}, "artifacts": []}
    threading.Thread(target=process_booking, args=(task_id, text), daemon=True).start()
    return jsonify(TASKS[task_id]), 202

@app.get("/a2a/tasks/<task_id>")
def tasks_get(task_id):
    task = TASKS.get(task_id)
    return jsonify(task if task else {"error": "Task not found"}), 200 if task else 404

# ── 4. Launch with make_server (gives us a handle to shut down later) ───
_httpd = make_server("0.0.0.0", SERVER_PORT, app)

def _run_server():
    _httpd.serve_forever()

server_thread = threading.Thread(target=_run_server, daemon=True)
server_thread.start()

# Verify the server is accepting connections
for attempt in range(10):
    try:
        with socket.create_connection(("localhost", SERVER_PORT), timeout=1):
            print(f"A2A server is running on port {SERVER_PORT} (attempt {attempt + 1})")
            break
    except OSError:
        time.sleep(0.5)
else:
    print("WARNING: Server did not start within 5 seconds.")

A2A server is running on port 8000 (attempt 1)


### Cell 2: A2A Client — Discover, Delegate, Poll
*(Slides 69–70)*

The server is running in a background thread. `SERVER_PORT` is read from
the kernel namespace so it always matches whichever port the server landed on.

In [ ]:
import requests, uuid, time, json

AGENT_BASE = f"http://localhost:{SERVER_PORT}"

# Health check: make sure the server is reachable before proceeding
try:
    health = requests.get(f"{AGENT_BASE}/.well-known/agent.json", timeout=3)
    health.raise_for_status()
except requests.exceptions.ConnectionError:
    raise RuntimeError(
        f"Cannot connect to A2A server on port {SERVER_PORT}. "
        "Please re-run the server cell (Cell 1) above first."
    )

# Step 1: Discover the agent's capabilities
card = health.json()
print(f"Agent: {card['name']}")
print(f"Skills: {[s['id'] for s in card['skills']]}")

# Step 2: Submit the task
task_text = ("Please confirm a venue in Edinburgh for 160 guests tonight. "
             "We require vegan catering and a quiet area for a webinar segment.")
task_id = str(uuid.uuid4())
task = requests.post(f"{AGENT_BASE}/a2a/tasks/send", json={
    "id": task_id,
    "message": {"role": "user",
                "parts": [{"type": "text", "text": task_text}]}
}).json()
print(f"\nTask {task_id[:8]}\u2026 submitted. State: {task['status']['state']}")

# Step 3: Poll until terminal state
print(f"Monitoring Task: {task_id[:8]}...")
for i in range(15):
    time.sleep(1)
    task = requests.get(f"{AGENT_BASE}/a2a/tasks/{task_id}").json()
    state = task["status"]["state"]
    print(f" [{i+1}] state={state}")
    if state in {"completed", "failed", "cancelled"}:
        break

# Step 4: Extract the artifact
result = "No artifact returned."
for artifact in task.get("artifacts", []):
    for part in artifact.get("parts", []):
        if part.get("type") == "text":
            result = part["text"]
            break

print("\n--- Final Result ---")
print(result)

INFO:werkzeug:127.0.0.1 - - [28/Mar/2026 23:14:47] "GET /.well-known/agent.json HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [28/Mar/2026 23:14:47] "POST /a2a/tasks/send HTTP/1.1" 202 -


Agent: EdinburghVenueBookingAgent
Skills: ['book_venue']

Task 9888ec13… submitted. State: submitted
Monitoring Task: 9888ec13...


INFO:werkzeug:127.0.0.1 - - [28/Mar/2026 23:14:48] "GET /a2a/tasks/9888ec13-0e91-431a-a6f0-0ecece06c5c4 HTTP/1.1" 200 -


 [1] state=completed

--- Final Result ---
CONFIRMED: The Haymarket Vaults — 160 guests, vegan menu included. Deposit: £200 required by 5 PM today. Ref: HV-2603.
